# Workflow Routing Tests

Simple, interactive tests for the LangGraph Agent Workflow.

In [ ]:
import sys
from pathlib import Path
from unittest.mock import patch

# Resolve project root
cwd = Path.cwd().resolve()
project_root = cwd.parents[1] if cwd.name == 'tests' else (cwd.parent if cwd.name == 'backend' else cwd)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.app.llm.base import BaseLLM
from backend.app.llm.llm_client import LLMClient
import backend.app.agents.graph.workflow as workflow_module
import backend.app.agents.graph.nodes.rag as rag_module

print("Setup completed.")

In [ ]:
class SimpleMockLLM(BaseLLM):
    """Simple Mock LLM for testing workflow routes."""
    def __init__(self, route: str):
        self.route = route

    async def generate(self, prompt: str) -> str:
        if "Classify the user request" in prompt:
            return f'{{"route": "{self.route}"}}'
        return f"Response for {self.route} route"

    async def stream(self, prompt: str):
        if False:
            yield prompt

print("SimpleMockLLM ready.")

## 1. Direct Route Test

In [ ]:
client = LLMClient(SimpleMockLLM("direct"))
workflow = workflow_module.create_workflow(client)

result = await workflow.ainvoke({"user_message": "Hello!"})

print("Output:", result["final_response"])
assert result["final_response"] == "Response for direct route"
print("✅ Direct Route Test Passed!")

## 2. Web Search Route Test

In [ ]:
client = LLMClient(SimpleMockLLM("web"))
workflow = workflow_module.create_workflow(client)

with patch.object(workflow_module, "search_web", return_value="Web search data"):
    result = await workflow.ainvoke({"user_message": "What is Python?"})

print("Output:", result["final_response"])
assert result["final_response"] == "Response for web route"
print("✅ Web Search Route Test Passed!")

## 3. Enterprise RAG Route Test

In [ ]:
client = LLMClient(SimpleMockLLM("rag"))
workflow = workflow_module.create_workflow(client)

mock_doc = [{"text": "Leave policy info", "metadata": {"source": "hr.pdf"}}]
with patch.object(rag_module, "retrieve", return_value=mock_doc):
    result = await workflow.ainvoke({"user_message": "What is annual leave?"})

print("Output:", result["final_response"])
assert result["final_response"] == "Response for rag route"
print("✅ Enterprise RAG Route Test Passed!")